# télos MDLM: Multi-Scale Training Suites
This notebook executes the training pipelines for:
1. **12.5M 1:30 Ratio** (376.2M Tokens, 2,870 steps)
2. **25M 1:35 Ratio** (875.0M Tokens, 6,676 steps — Resumed from step 5,000)

Memory GC and `mx.clear_cache()` are enforced to keep RAM usage strictly contained on Metal GPU.

In [ ]:
import os
import sys
import time
import gc
import yaml
import math
import io
from pathlib import Path
import numpy as np

# Ensure working directory is project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
import mlx.nn as nn
from telos.model.mlx_components import MLXTelosTransformer, load_upscaled_weights
from telos.training.trainer import TelosMLXTrainer
from telos.data.tokenizer import load_tokenizer

def run_training_step(config_path, upscaled_source=None, resume_from=None, resume_step=0):
    print("=" * 85)
    print("STARTING TRAINING RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if upscaled_source:
        src_ckpt, src_cfg = upscaled_source
        print("  [Net2Net] Upscaling model weights from: " + str(src_ckpt))
        load_upscaled_weights(model, cfg["model"], src_ckpt, src_cfg)
    
    if resume_from:
        print(f"  [Resume] Loading weights from {resume_from}")
        model.load_weights(resume_from, strict=False)
    
    trainer = TelosMLXTrainer(model, cfg)
    trainer.train(resume_step=resume_step)
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED RUN: " + str(config_path) + "\n")

In [ ]:
# PIPELINE DEFINITION: 25M 1:35 RESUME
start_time = time.time()

# 25M 1:35 (Resuming from step 5,000 -> 6,676, ~1.1 hrs remaining)
print("\n>>> Resuming 25M 1:35 from Step 5,000 <<<")
run_training_step(
    "configs/phase_b_25m_1to35_mlx.yaml",
    resume_from="checkpoints/phase_b_25m_1to35_mlx/checkpoint_step_5000.safetensors",
    resume_step=5000
)

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"ALL SCHEDULED RUNS COMPLETED SUCCESSFULLY IN {total_elapsed:.2f} HOURS!\n")
print("=" * 85)